# ERA5 nRMSE evaluation

Observation rates are 2, 3, 5, 8, 10, 15 and 20%, with five sampling seeds [1, 42, 123, 456, 789] and 50 reverse steps. The observation-mask seed is fixed while the sampling noise varies. Standard deviations use NumPy ddof=0.

Run cells in order. Sampling counts are reverse-diffusion steps; checkpoint numbers identify training state.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'models').is_dir() and (p / 'sampler').is_dir() and (p / 'trainer').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the MRSM repository.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT / 'outputs' / 'era5_nrmse'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.family': 'Arial', 'font.size': 10})

def format_current_figure():
    def tick(value, pos):
        return '0' if abs(value) < 1e-12 else f'{value:g}'
    for ax in plt.gcf().axes:
        if ax.get_xscale() == 'linear':
            ax.xaxis.set_major_formatter(FuncFormatter(tick))
        if ax.get_yscale() == 'linear':
            ax.yaxis.set_major_formatter(FuncFormatter(tick))

print('Repository:', ROOT)
print('Device:', device)


In [ ]:
import os
from sampler.utils import *
import importlib
import sampler.utils
from trainer.utils import restore_checkpoint
import torch
import torch.nn as nn
import json
from collections import namedtuple
from tqdm import tqdm
from models import get_model, get_ema
from sampler import VESDE
import numpy as np
from einops import rearrange
data_name = 'era5'
version = 'v0'
opt_path = './results/' + data_name + '_' + version + '/config.json'
with open(opt_path, 'r', encoding='utf-8') as f:
    opt = json.load(f)
    opt['num_channels'] = opt['num_components'] + opt['num_conditions']
opt['results_path'] = str(Path(opt_path).parent)
opt['device'] = device
OPT_class = namedtuple('OPT_class', opt.keys())
opt = OPT_class(**opt)
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'


In [ ]:
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
net = get_model(opt)
net.to(device)
ema = get_ema(net.parameters(), decay=opt.ema_rate)
optimizer = torch.optim.Adam(net.parameters(), lr=opt.lr, betas=(0.9, 0.999), eps=1e-08, weight_decay=0)
state = dict(optimizer=optimizer, model=net, ema=ema, epoch=0, iteration=0, loss_train=[])
ckpt_path = opt.results_path + '/checkpoint.pth'
if not Path(ckpt_path).is_file():
    raise FileNotFoundError(ckpt_path)
state = restore_checkpoint(ckpt_path, state, device)
net = state['model']
ema = state['ema']
ema.copy_to(net.parameters())
print('Model loaded from epoch', state['epoch'])


In [ ]:
data_test_path = './data/era5/era5_test.npy'
data_raw = np.load(data_test_path)
(_, z) = np.meshgrid(np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), np.linspace(0, 2 * np.pi, opt.image_size, endpoint=False), indexing='ij')
x_test = data_raw.transpose(0, 1, 4, 2, 3)
print(x_test.shape)


In [ ]:
m = 2
T = opt.num_frames
T_prime = 100
B = int(np.ceil((T_prime - m) / (T - m)))
T_prime = B * (T - m) + m
n_samples = 1
samples = []
seed_samples = 1
rnd = np.random.RandomState(seed_samples)
t_start = 0
samples = x_test[:, t_start:t_start + T_prime]
print(samples.shape)


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm import tqdm
import os
from contextlib import redirect_stdout, redirect_stderr
observable_portions_pct = [2, 3, 5, 8, 10, 15, 20]
observable_portions = [p / 100.0 for p in observable_portions_pct]
scales_to_run = [1.0 - op for op in observable_portions]
random_seeds = [1, 42, 123, 456, 789]
plot_data = {'observable_pct': observable_portions_pct, 'sde_u_mean': [], 'sde_u_std': [], 'sde_v_mean': [], 'sde_v_std': [], 'sde_t_mean': [], 'sde_t_std': [], 'sde_p_mean': [], 'sde_p_std': [], 'iss_u_mean': [], 'iss_u_std': [], 'iss_v_mean': [], 'iss_v_std': [], 'iss_t_mean': [], 'iss_t_std': [], 'iss_p_mean': [], 'iss_p_std': []}
true_data_for_rmse = samples.squeeze(0)
for (i, scale) in enumerate(tqdm(scales_to_run, desc='Processing Scales', dynamic_ncols=True, leave=True)):
    observable_pct = observable_portions_pct[i]
    results_at_this_scale = {'sde_u': [], 'sde_v': [], 'sde_t': [], 'sde_p': [], 'iss_u': [], 'iss_v': [], 'iss_t': [], 'iss_p': []}
    for seed_sample in tqdm(random_seeds, desc=f'  Seeds (scale {scale:.2f})', dynamic_ncols=True, leave=False):
        outer_loop = 50
        denoising_steps = outer_loop
        obs_type = 'random'
        alpha_case = 0.1
        beta = 0.4
        gamma = 0.5
        sde = VESDE(opt, sigma_min=opt.beta_min, sigma_max=opt.beta_max, N=outer_loop)
        net.eval()
        reap_fn = lambda x, num: torch.stack([x for _ in range(num)])
        setup_seed(seed_sample)
        with torch.no_grad():
            if obs_type.lower() == 'random':
                T_prime_y = T_prime
                mask = np.zeros([opt.num_components + opt.num_conditions, opt.image_size, opt.image_size])
                mask[:opt.num_components] = mask_gen([1, opt.image_size, opt.image_size], scale, seed=seed_samples)
                mask_torch = torch.from_numpy(mask).to(device)
                transform_gen = transform = lambda x: x[:, :T_prime_y] * mask[np.newaxis, np.newaxis] if isinstance(x, np.ndarray) else x[:, :T_prime_y] * mask_torch[None, None].to(x.device)
                sparsity = scale
            alpha = alpha_case / np.sqrt(1 - sparsity)
            samples_scale = samples.copy()
            y = transform_gen(samples_scale)
            latent_mask = torch.ones([T, 1, 1, 1]).float().to(device)
            obs_mask = torch.zeros([T, 1, 1, 1]).float().to(device)
            frame_indices = torch.arange(T).to(device)
            net_fn = lambda x, t: net(x, x0=x, timesteps=t, latent_mask=reap_fn(latent_mask, len(x)), obs_mask=reap_fn(obs_mask, len(x)), frame_indices=reap_fn(frame_indices, len(x)))[0]
            with open(os.devnull, 'w') as fnull:
                with redirect_stdout(fnull), redirect_stderr(fnull):
                    x_inpainted = s3gm_sample_2d(opt, net_fn, sde, y, transform_gen, alpha=alpha, beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
                    setup_seed(seed_sample)
                    x_inpainted1 = s3gm_sample_2d_ISS_ERA5(opt, net_fn, sde, y, transform_gen, alpha=0.3 / np.sqrt(1 - sparsity), beta=beta, gamma=gamma, T_prime_y=T_prime_y, T_prime=T_prime, overlap=m, corrector=NoneCorrector, n_steps=0, eta=0, reg_coef_pa=0, reg_coef_ar=0, denoising_steps=denoising_steps, snr=0.128, device=device, dtype='float', eps=1e-12, probability_flow=False, continuous=True)
        x_pred = x_inpainted
        x_pred1 = x_inpainted1
        pred_data_sde = x_pred.squeeze(0)
        pred_data_iss = x_pred1.squeeze(0)
        err_sde_u = np.mean(cal_rmse(true_data_for_rmse[:, 0:1], pred_data_sde[:, 0:1], normalize=True))
        err_sde_v = np.mean(cal_rmse(true_data_for_rmse[:, 1:2], pred_data_sde[:, 1:2], normalize=True))
        err_sde_t = np.mean(cal_rmse(true_data_for_rmse[:, 2:3], pred_data_sde[:, 2:3], normalize=True))
        err_sde_p = np.mean(cal_rmse(true_data_for_rmse[:, 3:4], pred_data_sde[:, 3:4], normalize=True))
        err_iss_u = np.mean(cal_rmse(true_data_for_rmse[:, 0:1], pred_data_iss[:, 0:1], normalize=True))
        err_iss_v = np.mean(cal_rmse(true_data_for_rmse[:, 1:2], pred_data_iss[:, 1:2], normalize=True))
        err_iss_t = np.mean(cal_rmse(true_data_for_rmse[:, 2:3], pred_data_iss[:, 2:3], normalize=True))
        err_iss_p = np.mean(cal_rmse(true_data_for_rmse[:, 3:4], pred_data_iss[:, 3:4], normalize=True))
        results_at_this_scale['sde_u'].append(err_sde_u)
        results_at_this_scale['sde_v'].append(err_sde_v)
        results_at_this_scale['sde_t'].append(err_sde_t)
        results_at_this_scale['sde_p'].append(err_sde_p)
        results_at_this_scale['iss_u'].append(err_iss_u)
        results_at_this_scale['iss_v'].append(err_iss_v)
        results_at_this_scale['iss_t'].append(err_iss_t)
        results_at_this_scale['iss_p'].append(err_iss_p)
        torch.cuda.empty_cache()
    for key in ['sde_u', 'sde_v', 'sde_t', 'sde_p', 'iss_u', 'iss_v', 'iss_t', 'iss_p']:
        plot_data[f'{key}_mean'].append(np.mean(results_at_this_scale[key]))
        plot_data[f'{key}_std'].append(np.std(results_at_this_scale[key]))


## Save aggregate metrics


In [ ]:
pd.DataFrame(plot_data).to_csv(OUTPUT / 'nrmse_by_observation_rate.csv', index=False)
print(pd.DataFrame(plot_data).to_string(index=False))
